# Praktikum Machine Learning — Bab 4
## Feature Engineering: Konstruksi, Seleksi Fitur, dan Pipeline Scikit-Learn

| | |
|---|---|
| **Nama** | Ahmad Dandi Subhani |
| **NPM** | 202343500126 |
| **Kelas** | R7B |
| **Mata Kuliah** | Machine Learning |
| **Dosen** | Nurfidah Dwitiyanti, M.Si. |

*Notebook ini dikerjakan di Google Colab / Jupyter Notebook.*

## Ringkasan Konsep

**Feature construction (konstruksi fitur)** adalah pembuatan fitur *baru* dari fitur mentah agar pola yang sulit dilihat model menjadi eksplisit — misalnya rasio (`AveBedrms / AveRooms`), interaksi antar-fitur, atau *binning* (mengubah numerik menjadi kategorikal seperti "wilayah" dari garis lintang). Berbeda dengan *feature extraction* (mis. PCA) yang menghasilkan fitur tak-bernama, hasil konstruksi fitur tetap bisa diinterpretasi manusia.

**Feature selection (seleksi fitur)** — memilih subset fitur paling relevan — punya tiga pendekatan:
1. **Filter** — menilai tiap fitur dengan skor statistik *tanpa* melibatkan model (contoh: `SelectKBest` dengan skor chi-square). Cepat, tetapi mengabaikan interaksi antar-fitur.
2. **Wrapper** — mengevaluasi subset fitur dengan melatih model sungguhan (contoh: **RFE**, *Recursive Feature Elimination*, yang membuang fitur terlemah satu per satu). Lebih akurat, tetapi mahal secara komputasi.
3. **Embedded** — seleksi terjadi *selama* pelatihan lewat regularisasi (contoh: **Lasso/L1** yang memaksa koefisien fitur tak penting menjadi nol). Kompromi antara kecepatan filter dan akurasi wrapper.

**ColumnTransformer** menerapkan transformasi *berbeda* ke kelompok kolom berbeda dalam satu langkah: mis. `StandardScaler` untuk kolom numerik, `OneHotEncoder` untuk kategorikal nominal, dan `OrdinalEncoder` untuk kategorikal ordinal — semua berjalan sekaligus pada satu DataFrame campuran.

**Pipeline** merangkai preprocessing + model menjadi **satu objek**. Keuntungan utamanya: `pipeline.fit(X_train)` hanya mempelajari statistik (mean, median, kategori) dari data latih, dan `pipeline.predict(data_baru)` otomatis menerapkan preprocessing yang *persis sama*. Ini menghilangkan risiko inkonsistensi antara preprocessing saat training dan saat deployment.

**Data leakage** adalah "kebocoran" informasi data uji ke proses training — misalnya mem-*fit* scaler/imputer pada *seluruh* data *sebelum* train-test split, sehingga statistik data uji ikut dipakai melatih model. Akibatnya skor evaluasi terlihat bagus padahal palsu. Pipeline mencegahnya karena setiap transformasi hanya di-fit pada data latih.

## 4.10–4.14 Implementasi Python

Pada bab ini kita membangun **pipeline prediksi harga rumah end-to-end** dengan dataset **California Housing** (data real dari scikit-learn): dari konstruksi fitur, preprocessing kolom campuran dengan `ColumnTransformer`, training dalam `Pipeline`, evaluasi, sampai visualisasi. Setiap tahap diikuti penjelasan outputnya.

### Praktikum 4.1 — Persiapan Library

Pertama, kita memuat semua library yang dibutuhkan bab ini: pandas/numpy untuk data, `ColumnTransformer` + `Pipeline` + transformer untuk preprocessing, `SelectKBest`/`chi2`/`RFE`/`Lasso` untuk seleksi fitur, serta `RandomForestRegressor` dan metrik evaluasi.

In [1]:
# Menjalankan di Google Colab / Jupyter Notebook
%matplotlib inline
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, chi2, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score
from scipy.stats import chi2_contingency, chi2 as chi2_dist

print("Semua library berhasil dimuat.")

Semua library berhasil dimuat.


**Penjelasan output:**
1. Tidak ada error `ModuleNotFoundError`, artinya seluruh library (`pandas`, `numpy`, `matplotlib`, `scikit-learn`, `scipy`) tersedia di environment ini.
2. Baris terakhir mencetak "Semua library berhasil dimuat." sebagai konfirmasi.

### Praktikum 4.2 — Memuat Dataset California Housing

Dataset California Housing dimuat langsung dari scikit-learn (`fetch_california_housing(as_frame=True)`) sehingga berbentuk DataFrame pandas. **Catatan penting:** versi scikit-learn dataset ini **tidak** memiliki kolom `OceanProximity` (itu hanya ada di versi seaborn) — yang tersedia hanyalah **8 fitur numerik** (`MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude`) plus target `MedHouseVal` (median harga rumah dalam ratusan ribu USD).

In [2]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Ukuran dataset:", df.shape)
print("\nInfo dataset:")
print(df.info())
print("\n5 baris pertama:")
print(df.head())
print("\nStatistik deskriptif:")
print(df.describe().T[['mean', 'std', 'min', 'max']])
print("\nJumlah missing value per kolom:")
print(df.isna().sum())

Ukuran dataset: (20640, 9)

Info dataset:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None

5 baris pertama:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   

**Penjelasan output:**
1. Dataset berukuran **(20640, 9)** — 20.640 blok sensus California dengan 8 fitur numerik + 1 target (`MedHouseVal`).
2. `df.info()` menunjukkan semua kolom bertipe `float64` dan **tidak ada missing value** (non-null = 20640 di semua kolom). `SimpleImputer` pada praktikum berikutnya tetap dipasang sebagai pengaman agar pipeline tahan terhadap data baru yang mungkin mengandung NaN.
3. `df.describe()` menunjukkan skala fitur sangat berbeda: `MedInc` (pendapatan median, rata-rata ~3,87) vs `Population` (rata-rata ~1425) — inilah alasan `StandardScaler` dibutuhkan agar model tidak didominasi fitur berskala besar.
4. `MedHouseVal` maksimum tepat **5.0** — nilai ini di-*clip* (dibatasi) oleh pembuat dataset, artinya harga di atas $500.000 dicatat sebagai 5.0.

### Praktikum 4.11 — Feature Construction & ColumnTransformer

Kita membuat tiga fitur konstruksi yang lebih informatif bagi model:
- `rooms_per_household` = `AveRooms` (rata-rata kamar per rumah tangga — dipakai langsung sebagai fitur eksplisit),
- `bedrooms_ratio` = `AveBedrms / AveRooms` (proporsi kamar tidur dari total kamar),
- `population_per_household` = `Population / AveOccup` (kepadatan hunian).

Karena dataset ini murni numerik, fitur **kategorikal** dibuat lewat *binning* agar `ColumnTransformer` untuk tipe kolom campuran tetap terdemonstrasi:
- `wilayah` (nominal): `Latitude` dipotong menjadi **Selatan / Tengah / Utara**,
- `kondisi_bangunan` (ordinal): `HouseAge` dipotong menjadi **Tua < Sedang < Baru** (ada urutan yang bermakna, sehingga dipakai `OrdinalEncoder` dengan urutan kategori eksplisit — bukan `OneHotEncoder`).

`ColumnTransformer` kemudian menerapkan: numerik → `SimpleImputer(median)` + `StandardScaler`; `wilayah` → `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')`; `kondisi_bangunan` → `OrdinalEncoder`.

In [3]:
# --- Feature construction: fitur rasio yang lebih informatif ---
df['rooms_per_household'] = df['AveRooms']
df['bedrooms_ratio'] = df['AveBedrms'] / df['AveRooms']
df['population_per_household'] = df['Population'] / df['AveOccup']

# --- Binning menjadi fitur kategorikal (pengganti OceanProximity yang tidak ada) ---
df['wilayah'] = pd.cut(df['Latitude'],
                       bins=[32.0, 34.5, 37.0, 42.0],
                       labels=['Selatan', 'Tengah', 'Utara'])
df['kondisi_bangunan'] = pd.Categorical(
    pd.cut(df['HouseAge'], bins=[0, 20, 40, 60], labels=['Tua', 'Sedang', 'Baru']),
    categories=['Tua', 'Sedang', 'Baru'], ordered=True)

print("Distribusi wilayah:", df['wilayah'].value_counts().to_dict())
print("Distribusi kondisi_bangunan:", df['kondisi_bangunan'].value_counts().to_dict())

# --- ColumnTransformer untuk kolom campuran ---
num_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
                'AveOccup', 'Latitude', 'Longitude',
                'rooms_per_household', 'bedrooms_ratio', 'population_per_household']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())]), num_features),
    ('wilayah', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), ['wilayah']),
    ('kondisi', OrdinalEncoder(categories=[['Tua', 'Sedang', 'Baru']]), ['kondisi_bangunan']),
])

X_cek = preprocessor.fit_transform(df.drop(columns=['MedHouseVal']))
print("\nBentuk matriks hasil transformasi:", X_cek.shape)
print("(11 numerik ter-scaling + 3 one-hot wilayah + 1 ordinal kondisi = 15 fitur)")
print("Nama fitur hasil transformasi:", preprocessor.get_feature_names_out().tolist())

Distribusi wilayah: {'Selatan': 10952, 'Utara': 7558, 'Tengah': 2130}
Distribusi kondisi_bangunan: {'Sedang': 10469, 'Tua': 6293, 'Baru': 3878}



Bentuk matriks hasil transformasi: (20640, 15)
(11 numerik ter-scaling + 3 one-hot wilayah + 1 ordinal kondisi = 15 fitur)
Nama fitur hasil transformasi: ['num__MedInc', 'num__HouseAge', 'num__AveRooms', 'num__AveBedrms', 'num__Population', 'num__AveOccup', 'num__Latitude', 'num__Longitude', 'num__rooms_per_household', 'num__bedrooms_ratio', 'num__population_per_household', 'wilayah__wilayah_Selatan', 'wilayah__wilayah_Tengah', 'wilayah__wilayah_Utara', 'kondisi__kondisi_bangunan']


**Penjelasan output:**
1. Binning menghasilkan distribusi yang masuk akal: wilayah **Selatan** terbanyak (California Selatan memang terpadat), dan kondisi bangunan terbanyak adalah **Sedang** (usia 20–40 tahun) — tidak ada kategori yang kosong, jadi encoder tidak akan menemui kategori tak dikenal saat training.
2. Matriks hasil transformasi berukuran **(20640, 15)**: 11 fitur numerik yang sudah di-*scaling*, 3 kolom one-hot (`wilayah_Selatan/Tengah/Utara`), dan 1 kolom ordinal (`kondisi_bangunan` bernilai 0/1/2 sesuai urutan Tua–Sedang–Baru).
3. `get_feature_names_out()` mengonfirmasi urutan kolom output — urutan inilah yang "dihafal" pipeline sehingga prediksi data baru selalu konsisten.

### Praktikum 4.12 — Training Pipeline

Target `y` adalah `MedHouseVal`. Data dibagi 80% latih / 20% uji dengan `random_state=42`. Kemudian preprocessing dan `RandomForestRegressor(n_estimators=200, random_state=42)` dirangkai dalam **satu Pipeline** dan di-fit sekaligus — inilah inti bab ini: model tidak pernah melihat data mentah, hanya data yang sudah melalui preprocessing yang sama persis.

In [4]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

pipe = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

t0 = time.time()
pipe.fit(X_train, y_train)
waktu_latih = time.time() - t0
print(f"Training selesai dalam {waktu_latih:.1f} detik")
print("Langkah pipeline:", [nama for nama, _ in pipe.steps])

Training selesai dalam 33.7 detik
Langkah pipeline: ['preprocessing', 'model']


**Penjelasan output:**
1. Training 200 pohon pada ~16.500 baris data latih selesai dalam **±34 detik** (dengan `n_jobs=-1` memanfaatkan semua core CPU) — sesuai harapan modul, tidak sampai 5 menit sehingga `n_estimators=200` dipertahankan.
2. Pipeline terdiri dari dua langkah berurutan: `preprocessing` → `model`. Karena keduanya dibungkus satu objek, tidak ada celah di mana data uji bisa "bocor" ke tahap preprocessing (data leakage) — scaler/imputer/encoder hanya di-fit pada `X_train`.

### Praktikum 4.13 — Evaluasi Model

Model dievaluasi pada data uji yang **belum pernah dilihat** selama training, memakai metrik **R²** (koefisien determinasi): 1.0 berarti sempurna, 0 berarti sebagus menebak rata-rata.

In [5]:
y_pred = pipe.predict(X_test)
skor_r2 = r2_score(y_test, y_pred)
print(f"R2 score pada data uji: {skor_r2:.4f}")
print(f"Artinya model menjelaskan {skor_r2*100:.1f}% variansi harga rumah pada data uji.")

R2 score pada data uji: 0.8093
Artinya model menjelaskan 80.9% variansi harga rumah pada data uji.


**Penjelasan output:**
1. R² pada data uji = **0,8093** — model menjelaskan **80,9%** variansi harga rumah. Ini skor yang kuat untuk dataset real yang noisy.
2. Karena evaluasi memakai data uji murni (bukan data latih), angka ini adalah estimasi performa yang jujur, bukan hasil overfitting.

### Praktikum 4.14 — Visualisasi Prediksi vs Aktual

Scatter plot **Harga Aktual (x)** vs **Harga Prediksi (y)**. Model yang baik menghasilkan titik-titik yang merapat ke **garis diagonal** (garis y = x, tempat prediksi = aktual).

In [6]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.15, s=10, label='Data uji')
maks = max(y_test.max(), y_pred.max())
plt.plot([0, maks], [0, maks], 'r--', linewidth=2, label='Garis ideal (prediksi = aktual)')
plt.xlabel('Harga Aktual (MedHouseVal)')
plt.ylabel('Harga Prediksi (MedHouseVal)')
plt.title('Prediksi vs Aktual (Pipeline Lengkap)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Penjelasan output:**
1. Mayoritas titik berkumpul rapat di sekitar garis diagonal merah — prediksi model dekat dengan harga aktual, konsisten dengan R² ±0,81.
2. Terlihat **garis horizontal di y = 5.0**: itu efek *clipping* target pada dataset (harga > $500rb dicatat 5.0) — model memprediksi nilai kontinu sehingga titik-titiknya "menggantung" di bawah/di atas garis tersebut. Ini keterbatasan data, bukan kesalahan model.
3. Penyebaran melebar pada harga tinggi (> 3.5): rumah mahal lebih sulit diprediksi karena sampelnya sedikit dan faktor penentunya lebih kompleks.

## Eksperimen (Coba-Coba)

Tiga percobaan untuk menguji pemahaman: (A) apa risiko preprocessing manual di luar pipeline, (B) apakah feature construction benar-benar membantu, dan (C) bagaimana trade-off jumlah pohon vs waktu latih.

### Eksperimen A — Pipeline vs Preprocessing Manual

Skenario deployment: model sudah dilatih, datang **1 baris data baru** (fitur mentah). Kita bandingkan tiga cara memprediksinya:
1. `pipe.predict(data_baru)` — cara pipeline (satu baris kode),
2. memakai **model saja** pada data mentah (jalan pintas yang sering dicoba),
3. preprocessing **manual** yang direplikasi tangan — versi benar vs versi yang kolom one-hot-nya tertukar urutannya.

In [7]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Pilih 1 baris data baru yang wilayahnya 'Selatan' -> one-hot [1,0,0],
# sehingga kalau kolomnya tertukar ([0,0,1]) efeknya terlihat jelas.
idx_selatan = X_test[X_test['wilayah'] == 'Selatan'].index[0]
baris_baru = X_test.loc[[idx_selatan]]
model_saja = pipe.named_steps['model']

print("1) pipeline.predict :", pipe.predict(baris_baru)[0])

# 2) Jalan pintas: model langsung pada data mentah
try:
    model_saja.predict(baris_baru)
except Exception as e:
    print("2) model saja      : GAGAL ->", type(e).__name__, ":", str(e)[:110])

# 3) Preprocessing manual: replikasi transformasi tangan memakai transformer ter-fit
pre = pipe.named_steps['preprocessing']
num_t = pre.named_transformers_['num'].transform(baris_baru[num_features])
oh_t  = pre.named_transformers_['wilayah'].transform(baris_baru[['wilayah']])
ord_t = pre.named_transformers_['kondisi'].transform(baris_baru[['kondisi_bangunan']])
manual_benar = np.hstack([num_t, oh_t, ord_t])
manual_salah = np.hstack([num_t, oh_t[:, ::-1], ord_t])  # kolom one-hot tertukar!

p_benar = model_saja.predict(manual_benar)[0]
p_salah = model_saja.predict(manual_salah)[0]
print("3a) manual (benar)             :", p_benar)
print("3b) manual (one-hot tertukar)  :", p_salah)
print("   -> selisih akibat tertukar  : %.4f" % abs(p_benar - p_salah))

1) pipeline.predict : 2.543115
2) model saja      : GAGAL -> ValueError : could not convert string to float: 'Selatan'
3a) manual (benar)             : 2.543115
3b) manual (one-hot tertukar)  : 2.532279999999999
   -> selisih akibat tertukar  : 0.0108


**Penjelasan output:**
1. `pipe.predict` langsung menghasilkan prediksi yang benar dari data mentah (**2,5431**) — seluruh preprocessing (imputasi, scaling, one-hot, ordinal) diterapkan otomatis dalam urutan yang tepat.
2. Memakai **model saja** pada data mentah langsung **GAGAL** (`ValueError: could not convert string to float: 'Selatan'`): model mengharapkan 15 fitur hasil transformasi numerik, tetapi menerima 11 kolom mentah yang masih berisi string. Ini membuktikan preprocessing bukan opsional — tanpanya model tidak bisa dipakai sama sekali.
3. Replikasi **manual yang benar** menghasilkan angka *persis sama* dengan pipeline (**2,5431**), tetapi butuh banyak baris kode dan pengetahuan detail (nama tiap transformer, urutan kolom). Versi **manual yang salah** (dua kolom one-hot `wilayah` tertukar urutannya) tetap berjalan **tanpa error** namun menghasilkan **2,5323** — selisih **0,0108** dari jawaban yang benar. Inilah bahaya sesungguhnya: kesalahan preprocessing manual sering *silent* (tidak menimbulkan error) sehingga lolos ke produksi tanpa disadari. Pipeline menghilangkan seluruh kelas bug ini.

### Eksperimen B — Dengan vs Tanpa Feature Construction

Apakah fitur hasil konstruksi (3 rasio + `wilayah` + `kondisi_bangunan`) benar-benar membantu? Kita latih pipeline kedua yang **hanya** memakai 8 fitur numerik asli (tetap dengan imputasi + scaling) memakai RandomForest yang sama, lalu bandingkan R².

In [8]:
pipe_tanpa = Pipeline(steps=[
    ('preprocessing', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]),
         ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
          'AveOccup', 'Latitude', 'Longitude']) ])),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
pipe_tanpa.fit(X_train, y_train)
r2_tanpa = r2_score(y_test, pipe_tanpa.predict(X_test))

print(f"R2 tanpa feature construction : {r2_tanpa:.4f}")
print(f"R2 dengan feature construction : {skor_r2:.4f}")
print(f"Selisih                         : {skor_r2 - r2_tanpa:+.4f}")

R2 tanpa feature construction : 0.8060
R2 dengan feature construction : 0.8093
Selisih                         : +0.0033


**Penjelasan output:**
1. Pipeline **dengan** feature construction mencapai R² **0,8093**, sedikit **lebih tinggi** daripada **0,8060** pada pipeline yang hanya memakai 8 fitur asli (selisih **+0,0033**) — fitur rasio dan hasil binning memberi sinyal tambahan yang berguna bagi RandomForest.
2. Kenaikannya kecil (bukan lompatan besar): wajar, karena RandomForest sudah pandai menangkap interaksi antar-fitur secara implisit. Nilai konstruksi fitur justru paling terasa pada model linear yang tidak bisa menemukan interaksi sendiri — dan pada keterbacaan: `bedrooms_ratio` jauh lebih mudah diinterpretasi stakeholder daripada interaksi implisit di dalam 200 pohon.
3. Kesimpulan: feature construction **membantu, tetapi bukan sulap** — selalu ukur dengan metrik, jangan berasumsi.

### Eksperimen C — n_estimators = 50 vs 200

Berapa harga yang dibayar untuk 200 pohon dibanding 50 pohon? Kita latih pipeline lengkap dengan kedua nilai tersebut dan bandingkan **R²** serta **waktu latih**.

In [9]:
hasil = {}
for n in [50, 200]:
    p = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('model', RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)),
    ])
    t0 = time.time()
    p.fit(X_train, y_train)
    dt = time.time() - t0
    r2 = r2_score(y_test, p.predict(X_test))
    hasil[n] = (r2, dt)
    print(f"n_estimators={n:3d} -> R2 = {r2:.4f}, waktu latih = {dt:5.1f} detik")

r2_50, t_50 = hasil[50]; r2_200, t_200 = hasil[200]
print(f"\nTambahan 150 pohon: R2 naik {r2_200 - r2_50:+.4f} dengan waktu {t_200/t_50:.1f}x lebih lama.")

n_estimators= 50 -> R2 = 0.8066, waktu latih =  10.9 detik


n_estimators=200 -> R2 = 0.8093, waktu latih =  34.5 detik

Tambahan 150 pohon: R2 naik +0.0028 dengan waktu 3.2x lebih lama.


**Penjelasan output:**
1. Waktu latih naik mendekati linear terhadap jumlah pohon: **10,9 detik** (50 pohon) vs **34,5 detik** (200 pohon) — **3,2x lebih lama** — tetapi R² hanya naik dari **0,8066** ke **0,8093** (**+0,0028**). Inilah *diminishing return*: 50 pohon pertama menangkap sebagian besar pola, 150 pohon tambahan hanya menghaluskan sedikit.
2. Pelajaran praktis (*trade-off*): jika waktu/komputasi terbatas (mis. tuning hyperparameter yang melatih puluhan model), `n_estimators=50`–`100` memberi >99% performa dengan sepertiga waktu. 200 pohon masuk akal untuk model final yang dilatih sekali.

## 4.8 Studi Kasus — Pipeline Prediksi Harga Rumah

| | |
|---|---|
| **Masalah** | Prediksi harga rumah dari dataset campuran: numerik (luas, jumlah kamar), kategorikal (tipe properti, kota), ordinal (kondisi bangunan: buruk–sangat baik). Preprocessing manual rawan inkonsisten antara fase training dan prediksi data baru. |
| **Solusi & proses** | (1) Feature construction: rasio `luas_bangunan/luas_tanah` dan `usia_bangunan`; (2) `ColumnTransformer`: `StandardScaler` untuk numerik, `OneHotEncoder` untuk kategorikal, `OrdinalEncoder` untuk kondisi bangunan; (3) seleksi fitur embedded via Lasso; (4) semuanya + model dibungkus satu `Pipeline`. |
| **Hasil & interpretasi** | Pipeline lengkap **lebih stabil pada data baru** karena preprocessing yang dipakai saat prediksi *dijamin identik* dengan saat training. Risiko inkonsistensi preprocessing train-vs-prediksi **hilang** — `pipeline.predict()` cukup dipanggil pada data mentah, sehingga deployment (Bab 14) menjadi sederhana dan minim risiko salah. |

**Kaitan dengan praktikum di atas:** tepat seperti yang kita buktikan pada Eksperimen A — pipeline yang kita bangun untuk California Housing memprediksi 1 baris data baru dalam satu panggilan, sedangkan replikasi manual berisiko *silent error* (hasil salah tanpa pesan error). Studi kasus ini adalah versi "dunia nyata" dari pola yang sama.

## 4.9 Contoh Perhitungan — Chi-Square

**Soal (dari modul):** fitur kategorikal `metode_pembayaran` (Tunai/Kredit) vs target `churn` (Ya/Tidak), n = 100.

| | Churn = Ya | Churn = Tidak |
|---|---|---|
| **Tunai** | 10 (E=15) | 40 (E=35) |
| **Kredit** | 20 (E=15) | 30 (E=35) |

**Perhitungan manual:**
chi² = (10−15)²/15 + (40−35)²/35 + (20−15)²/15 + (30−35)²/35
= 25/15 + 25/35 + 25/15 + 25/35
= 1,667 + 0,714 + 1,667 + 0,714
= **4,76**

Derajat bebas db = (2−1)(2−1) = **1**. Nilai kritis chi-square untuk α = 0,05 dan db = 1 adalah **3,841**. Karena 4,76 > 3,841 → H₀ (tidak ada hubungan) **ditolak**: `metode_pembayaran` berhubungan signifikan dengan churn, sehingga fitur ini **relevan** dan layak dipertahankan. Prinsipnya: makin besar chi-square → indikasi fitur makin relevan.

**Verifikasi dengan kode** (`scipy.stats.chi2_contingency`):

In [10]:
observed = np.array([[10, 40],   # Tunai : Ya=10, Tidak=40
                     [20, 30]])  # Kredit: Ya=20, Tidak=30

stat, p_value, dof, expected = chi2_contingency(observed, correction=False)
manual = (10-15)**2/15 + (40-35)**2/35 + (20-15)**2/15 + (30-35)**2/35
kritis = chi2_dist.ppf(0.95, df=1)

print(f"chi2 manual (modul) : {manual:.4f}")
print(f"chi2 scipy          : {stat:.4f}")
print(f"db                  : {dof}")
print(f"p-value             : {p_value:.4f}")
print(f"nilai kritis (a=0.05, db=1): {kritis:.4f}")
print("Keputusan           :", "TOLAK H0 -> fitur RELEVAN" if stat > kritis else "GAGAL tolak H0")

chi2 manual (modul) : 4.7619
chi2 scipy          : 4.7619
db                  : 1
p-value             : 0.0291
nilai kritis (a=0.05, db=1): 3.8415
Keputusan           : TOLAK H0 -> fitur RELEVAN


**Penjelasan output:**
1. `chi2_contingency` menghasilkan **4,7619** — cocok dengan perhitungan manual modul (**4,76**); selisih hanya pembulatan. Nilai harapan (expected) yang dipakai scipy pun sama: 15/35/15/35.
2. p-value = **0,0291** < 0,05, konsisten dengan keputusan: statistik uji (**4,7619**) **melewati** nilai kritis (**3,8415**), jadi H₀ ditolak.
3. Interpretasi untuk feature selection: karena `metode_pembayaran` terbukti berhubungan signifikan dengan `churn`, metode **filter** seperti `SelectKBest(chi2)` akan memberi skor tinggi pada fitur ini dan mempertahankannya — tepat seperti logika yang dipakai pada Praktikum 1 di bawah.

## Latihan Praktikum

Dua latihan untuk mengonsolidasikan bab ini: (1) membandingkan tiga pendekatan feature selection pada dataset klasifikasi, dan (2) membangun pipeline lengkap kedua pada dataset campuran yang berbeda.

### Praktikum 1 (4.3) — Membandingkan Tiga Pendekatan Feature Selection

Dataset **Breast Cancer** (569 sampel, 30 fitur numerik — semuanya non-negatif sehingga skor chi-square valid). Tiga metode memilih 10 fitur terbaik (embedded memakai L1-logistic yang menyeleksi secara otomatis):
- **Filter:** `SelectKBest(chi2, k=10)` — skor statistik murni, tanpa model.
- **Wrapper:** `RFE(LogisticRegression, n_features_to_select=10)` — eliminasi rekursif memakai model.
- **Embedded:** `SelectFromModel(LogisticRegression(l1_ratio=1, C=0.1))` — regularisasi L1 memaksa koefisien tak penting menjadi nol *selama* training.

In [11]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
nama_fitur = load_breast_cancer().feature_names

# 1) Filter
skb = SelectKBest(chi2, k=10).fit(X_bc, y_bc)
f_filter = set(np.argsort(skb.scores_)[-10:])

# 2) Wrapper
rfe = RFE(LogisticRegression(max_iter=5000), n_features_to_select=10).fit(X_bc, y_bc)
f_wrapper = set(rfe.get_support(indices=True))

# 3) Embedded (L1)
l1 = SelectFromModel(
    LogisticRegression(l1_ratio=1, solver='saga', C=0.1, max_iter=10000, random_state=42)
).fit(X_bc, y_bc)
f_embedded = set(l1.get_support(indices=True))

semua = sorted(f_filter | f_wrapper | f_embedded)
tabel = pd.DataFrame({
    'fitur': [nama_fitur[i] for i in semua],
    'filter (chi2)': ['ya' if i in f_filter else '-' for i in semua],
    'wrapper (RFE)': ['ya' if i in f_wrapper else '-' for i in semua],
    'embedded (L1)': ['ya' if i in f_embedded else '-' for i in semua],
})
print(tabel.to_string(index=False))
print(f"\nJumlah fitur terpilih: filter={len(f_filter)}, wrapper={len(f_wrapper)}, embedded={len(f_embedded)}")
print(f"Irisan ketiga metode (dipilih semua): {[nama_fitur[i] for i in sorted(f_filter & f_wrapper & f_embedded)]}")

               fitur filter (chi2) wrapper (RFE) embedded (L1)
         mean radius            ya            ya            ya
        mean texture            ya             -            ya
      mean perimeter            ya             -            ya
           mean area            ya             -            ya
    mean compactness             -            ya             -
      mean concavity             -            ya             -
       texture error             -            ya             -
     perimeter error            ya             -             -
          area error            ya             -            ya
        worst radius            ya            ya            ya
       worst texture            ya             -            ya
     worst perimeter            ya             -            ya
          worst area            ya             -            ya
    worst smoothness             -            ya             -
   worst compactness             -            ya       

**Penjelasan output:**
1. Ketiga metode **tidak memilih himpunan yang identik** — dari 18 fitur yang dipilih minimal satu metode, hanya **2** yang disepakati ketiganya: `mean radius` dan `worst radius`. Ini wajar: filter menilai tiap fitur *sendiri-sendiri* sehingga cenderung memilih fitur-fitur yang saling berkorelasi (banyak varian "worst ..." dan "mean ..."), sedangkan wrapper dan embedded mempertimbangkan **kontribusi fitur dalam konteks model** — RFE justru banyak memilih fitur "*... error*" dan "*worst ...*" yang saling melengkapi di dalam model.
2. **Filter (chi2)** tercepat — hanya menghitung statistik sekali. **Wrapper (RFE)** paling lambat — melatih LogisticRegression berulang kali sambil membuang fitur terlemah. **Embedded (L1)** hanya butuh satu kali training dan otomatis menentukan jumlah fitur (di sini 9 fitur, bukan genap 10).
3. **Kesimpulan:** tidak ada metode yang "selalu benar". Filter cocok untuk penyaringan awal cepat pada data berdimensi besar; wrapper untuk akurasi maksimal saat komputasi bukan masalah; embedded sebagai jalan tengah yang elegan — seleksi terjadi sebagai efek samping training. Pada praktiknya ketiganya sering **dikombinasikan** (filter dulu untuk membuang yang jelas-jelas noise, lalu wrapper/embedded).

### Praktikum 2 — Membangun Pipeline Lengkap (Dataset Titanic)

Dataset **Titanic** (`seaborn.load_dataset('titanic')`) adalah contoh klasik data campuran: numerik (`age`, `fare`, `sibsp`, `parch`) + kategorikal (`sex`, `embarked`, `class`) dengan target klasifikasi `survived`. Pipeline: `ColumnTransformer` (numerik → median + scaler; kategorikal → most_frequent + one-hot) + `LogisticRegression`, dievaluasi dengan **akurasi** pada data uji.

> **Catatan fallback:** kode di bawah mencoba mengunduh dataset Titanic; jika unduhan gagal (tidak ada internet), otomatis dipakai dataset fallback — California Housing dengan target biner (`MedHouseVal` > median) + kolom kategorikal hasil binning (`wilayah`, `kondisi_bangunan`) — sehingga praktikum tetap berjalan dan konsep ColumnTransformer untuk tipe kolom campuran tetap terdemonstrasi.

In [12]:
import seaborn as sns

try:
    df_t = sns.load_dataset('titanic')
    print("Dataset Titanic berhasil dimuat dari seaborn:", df_t.shape)
    pakai_fallback = False
except Exception as e:
    print("Unduh Titanic gagal (%s). Memakai dataset fallback." % type(e).__name__)
    df_t = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
               'AveOccup', 'Latitude', 'Longitude',
               'wilayah', 'kondisi_bangunan']].copy()
    df_t['survived'] = (df['MedHouseVal'] > df['MedHouseVal'].median()).astype(int)
    df_t = df_t.rename(columns={'wilayah': 'sex', 'kondisi_bangunan': 'class'})
    df_t['embarked'] = np.where(df_t['Latitude'] > 36, 'S', 'C')
    df_t = df_t.rename(columns={'MedInc': 'fare', 'HouseAge': 'age',
                                'AveRooms': 'sibsp', 'AveBedrms': 'parch'})
    pakai_fallback = True
    print("Dataset fallback siap:", df_t.shape)

if not pakai_fallback:
    fitur_num_t = ['age', 'fare', 'sibsp', 'parch']
    fitur_cat_t = ['sex', 'embarked', 'class']
    X_t = df_t[fitur_num_t + fitur_cat_t]
    y_t = df_t['survived']
else:
    fitur_num_t = ['fare', 'age', 'sibsp', 'parch', 'Population', 'AveOccup', 'Longitude']
    fitur_cat_t = ['sex', 'embarked', 'class']
    X_t = df_t[fitur_num_t + fitur_cat_t]
    y_t = df_t['survived']

print("\nMissing value per kolom:", X_t.isna().sum().to_dict())

Xtr, Xte, ytr, yte = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

pipe_titanic = Pipeline(steps=[
    ('preprocessing', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]), fitur_num_t),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), fitur_cat_t),
    ])),
    ('model', LogisticRegression(max_iter=1000)),
])
pipe_titanic.fit(Xtr, ytr)
akurasi = accuracy_score(yte, pipe_titanic.predict(Xte))
print(f"\nAkurasi pada data uji: {akurasi:.4f}")
print("Prediksi 1 penumpang baru:", pipe_titanic.predict(Xte.iloc[[0]])[0],
      "(aktual:", yte.iloc[0], ")")

Dataset Titanic berhasil dimuat dari seaborn: (891, 15)

Missing value per kolom: {'age': 177, 'fare': 0, 'sibsp': 0, 'parch': 0, 'sex': 0, 'embarked': 2, 'class': 0}



Akurasi pada data uji: 0.7989
Prediksi 1 penumpang baru: 0 (aktual: 1 )


**Penjelasan output:**
1. Dataset Titanic berhasil dimuat (891 baris, 15 kolom) — **fallback tidak terpakai** di environment ini, tetapi logikanya siap jika notebook dijalankan offline.
2. Missing value terdeteksi pada `age` (±177) dan `embarked` (2) — keduanya ditangani otomatis di dalam pipeline (median untuk numerik, modus untuk kategorikal), tanpa satu baris pun kode pembersihan manual di luar pipeline.
3. Akurasi **0,7989** pada data uji: wajar untuk model linear sederhana tanpa *feature engineering* lanjutan — dan konsisten dengan benchmark umum dataset Titanic.
4. Prediksi satu baris data mentah langsung jalan tanpa preprocessing manual — sekali lagi menunjukkan keunggulan pipeline: data baru cukup dilempar ke `pipe.predict()`, preprocessing yang tepat (termasuk penanganan NaN dan kategori tak dikenal via `handle_unknown='ignore'`) sudah "terkunci" di dalamnya. Catatan jujur: prediksi contoh baris tersebut adalah 0 padahal aktualnya 1 — pengingat bahwa akurasi 80% berarti ~1 dari 5 prediksi meleset.

## Kesimpulan Bab 4

1. **Feature construction** membuat pola implisit menjadi eksplisit (rasio, binning): pada California Housing, fitur rasio + hasil binning (`wilayah`, `kondisi_bangunan`) menaikkan R² dibanding 8 fitur asli saja — selalu verifikasi dengan metrik, karena kenaikannya bisa kecil pada model seperti RandomForest yang sudah menangkap interaksi sendiri.
2. Tiga pendekatan **feature selection** — *filter* (cepat, statistik murni), *wrapper* (akurat, mahal), *embedded* (seleksi menyatu dengan training) — memilih himpunan fitur yang berbeda-beda; tidak ada yang selalu terbaik, dan ketiganya sering dikombinasikan.
3. **ColumnTransformer + Pipeline** adalah cara idiomatis scikit-learn menangani data campuran: tiap tipe kolom mendapat preprocessing yang tepat, semuanya di-fit *hanya* pada data latih sehingga **data leakage** terhindar.
4. Keuntungan terbesar Pipeline terlihat saat **deployment**: `pipeline.predict(data_mentah)` selalu menerapkan preprocessing yang identik dengan training — eksperimen kita membuktikan replikasi manual berisiko *silent error* (hasil salah tanpa pesan error), sedangkan pipeline menghilangkan seluruh kelas bug tersebut.
5. Selalu ada **trade-off komputasi vs performa**: 200 pohon vs 50 pohon menaikkan R² tipis dengan waktu ~4x lipat — pilih jumlah pohon sesuai anggaran komputasi, bukan sekadar "lebih besar lebih baik".